In [ ]:
!pip uninstall -y numpy matplotlib ultralytics opencv-python-headless

!pip install "numpy==1.26.4"

!pip install "matplotlib" "ultralytics" "opencv-python-headless"

In [ ]:
import numpy
numpy.__version__

In [ ]:
import os
import shutil
import random
from pycocotools.coco import COCO
from tqdm import tqdm
import yaml
# Paths
coco_annotation_path = '/kaggle/input/coco-2017-dataset/coco2017/annotations/instances_train2017.json'
coco_images_path = '/kaggle/input/coco-2017-dataset/coco2017/train2017'
output_dir = '/kaggle/working/custom_dataset'
images_per_class = 500

target_classes = [
    "person", "handbag", "backpack", "suitcase",
    "bicycle", "car", "motorcycle", "bus", "truck", "train", "airplane",
    "dog", "cat", "bird", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe",
    "chair", "couch", "bed", "toilet", "dining table",
    "tv", "laptop", "mouse", "keyboard", "cell phone",
    "bottle", "cup", "fork", "knife", "spoon", "bowl",
    "skis", "snowboard", "surfboard", "tennis racket", "baseball bat", "frisbee", "kite", "skateboard",
    "clock", "book", "umbrella"
]

# Create the set of classes we need to extract
needed_coco_classes = set(target_classes)

def create_dataset():
    # Initialize COCO api
    coco = COCO(coco_annotation_path)
    
    # Get ID for each category name
    # We explicitly convert the set to a list to avoid ordering issues
    cat_ids = coco.getCatIds(catNms=list(needed_coco_classes))
    cats = coco.loadCats(cat_ids)
    
    # Create a lookup: COCO_ID -> New_Class_ID (0, 1, 2...)
    cat_id_to_new_id = {cat['id']: i for i, cat in enumerate(cats)}
    new_id_to_name = {i: cat['name'] for i, cat in enumerate(cats)}
    
    # Prepare directories
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir) # Clean up previous runs
    os.makedirs(f"{output_dir}/images/train", exist_ok=True)
    os.makedirs(f"{output_dir}/labels/train", exist_ok=True)

    # Track how many images we have for each class
    class_counts = {cat['name']: 0 for cat in cats}
    processed_image_ids = set()

    print("Selecting images...")
    
    # Iterate through each category
    for cat in tqdm(cats):
        cat_name = cat['name']
        cat_id = cat['id']
        
        # Get all image IDs containing this category
        img_ids = coco.getImgIds(catIds=[cat_id])
        random.shuffle(img_ids) 
        
        for img_id in img_ids:
            # Stop if we have enough for this class
            if class_counts[cat_name] >= images_per_class:
                break
                
            # Skip if we already processed this image
            if img_id in processed_image_ids:
                continue

            # Load Image Info
            img_info = coco.loadImgs(img_id)[0]
            file_name = img_info['file_name']
            
            # COPY IMAGE
            src_img = os.path.join(coco_images_path, file_name)
            if not os.path.exists(src_img):
                continue
                
            dst_img = os.path.join(output_dir, "images/train", file_name)
            shutil.copy(src_img, dst_img)
            
            # GENERATE LABEL (YOLO FORMAT)
            ann_ids = coco.getAnnIds(imgIds=img_id, catIds=cat_ids)
            anns = coco.loadAnns(ann_ids)
            
            label_file = file_name.replace('.jpg', '.txt')
            label_path = os.path.join(output_dir, "labels/train", label_file)
            
            has_relevant_obj = False
            with open(label_path, 'w') as f:
                for ann in anns:
                    # BBox calculations
                    x_min, y_min, w, h = ann['bbox']
                    
                    img_w, img_h = img_info['width'], img_info['height']
                    x_center = (x_min + w / 2) / img_w
                    y_center = (y_min + h / 2) / img_h
                    w_norm = w / img_w
                    h_norm = h / img_h
                    
                    # Get the new class index (0-N)
                    class_idx = cat_id_to_new_id[ann['category_id']]
                    
                    f.write(f"{class_idx} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")
                    
                    # Update counts
                    current_cat_name = new_id_to_name[class_idx]
                    class_counts[current_cat_name] += 1
                    has_relevant_obj = True

            if has_relevant_obj:
                processed_image_ids.add(img_id)

    # CREATE DATA.YAML for RT-DETR
    
    
    data = {
        "path": output_dir,
        "train": "images/train",
        "val": "images/train",
        "nc": len(cats),
        "names": {i:  new_id_to_name[i] for i in range(len(cats))}
    }
    
    yaml_content = yaml.dump(data, sort_keys=False)
    
    print(yaml_content)
    
    with open(f"{output_dir}/data.yaml", "w") as f:
        f.write(yaml_content)


if __name__ == "__main__":
    create_dataset()

In [ ]:
from ultralytics import YOLO

# Load YOLO11n model
model = YOLO('yolo11s.pt')

# Train
results = model.train(
    data='/kaggle/working/custom_dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    project='/kaggle/working/runs/train',
    name='yolo11s_training',
    device=0,
    amp=False 
)

Ultralytics 8.3.235 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/custom_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11s_training, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, per

In [3]:
columns = ['metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)','metrics/mAP50-95(B)']
df.loc[49, columns]

metrics/precision(B)    0.90336
metrics/recall(B)       0.76169
metrics/mAP50(B)        0.84866
metrics/mAP50-95(B)     0.68142
Name: 49, dtype: float64